# Fase 2: Data Undestanding

En esta segunda etapa, el objetivo es realizar una primera inmersión en el dataset para comprender su volumen, la naturaleza de sus variables y la calidad inicial de la información recibida. Siguiendo la guía recibida, este es el paso previo indispensable antes de realizar cualquier modificación o limpieza profunda de los datos.

## 1. Exploratorio Inicial (First Look)
Obtendremos una visión general de las dimensiones del dataset y una muestra de los registros para entender con qué estamos trabajando.

In [5]:
# Carga de librerías
import pandas as pd

# Carga del dataset
df = pd.read_csv("../data/ai4i2020.csv")

# Visualización de dimensiones
print(f"Filas: {df.shape[0]:,} | Columnas: {df.shape[1]}")

# Primeros valores
df.head()

Filas: 10,000 | Columnas: 14


,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0


In [6]:
# Últimos valores
df.tail()

,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
9995,9996,M24855,M,298.8,308.4,1604,29.5,14,0,0,0,0,0,0
9996,9997,H39410,H,298.9,308.4,1632,31.8,17,0,0,0,0,0,0
9997,9998,M24857,M,299.0,308.6,1645,33.4,22,0,0,0,0,0,0
9998,9999,H39412,H,299.0,308.7,1408,48.5,25,0,0,0,0,0,0
9999,10000,M24859,M,299.0,308.7,1500,40.2,30,0,0,0,0,0,0


In [7]:
# Información y tipos
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   UDI                      10000 non-null  int64  
 1   Product ID               10000 non-null  object 
 2   Type                     10000 non-null  object 
 3   Air temperature [K]      10000 non-null  float64
 4   Process temperature [K]  10000 non-null  float64
 5   Rotational speed [rpm]   10000 non-null  int64  
 6   Torque [Nm]              10000 non-null  float64
 7   Tool wear [min]          10000 non-null  int64  
 8   Machine failure          10000 non-null  int64  
 9   TWF                      10000 non-null  int64  
 10  HDF                      10000 non-null  int64  
 11  PWF                      10000 non-null  int64  
 12  OSF                      10000 non-null  int64  
 13  RNF                      10000 non-null  int64  
dtypes: float64(3), int64(9)

In [8]:
# Missings por columna
df.isnull().sum().sort_values(ascending=False)

UDI                        0
Product ID                 0
Type                       0
Air temperature [K]        0
Process temperature [K]    0
Rotational speed [rpm]     0
Torque [Nm]                0
Tool wear [min]            0
Machine failure            0
TWF                        0
HDF                        0
PWF                        0
OSF                        0
RNF                        0
dtype: int64

In [13]:
# Duplicados
print(f"Filas duplicadas: {df.duplicated().sum()}")

Filas duplicadas: 0


## 2. Inspección Técnica de Variables (Data Dictionary)
Analizamos los tipos de datos asignados automáticamente por Python y detectamos la presencia de valores nulos iniciales.

In [9]:
# Generación del data dictionary automático
data_dict = pd.DataFrame({
    'columna': df.columns,
    'dtype': df.dtypes.values,
    'nulos_pct': (df.isnull().mean() * 100).round(1).values,
    'unicos': df.nunique().values,
    'ejemplo': [df[c].dropna().iloc[0] if len(df[c].dropna()) > 0 else None for c in df.columns]
})

display(data_dict)

,columna,dtype,nulos_pct,unicos,ejemplo
0,UDI,int64,0.0,10000,1
1,Product ID,object,0.0,10000,M14860
2,Type,object,0.0,3,M
3,Air temperature [K],float64,0.0,93,298.1
4,Process temperature [K],float64,0.0,82,308.6
5,Rotational speed [rpm],int64,0.0,941,1551
6,Torque [Nm],float64,0.0,577,42.8
7,Tool wear [min],int64,0.0,246,0
8,Machine failure,int64,0.0,2,0
9,TWF,int64,0.0,2,0


# Descripción del dataset: AI4I 2020 Predictive Maintenance

Dataset **sintético** que simula datos de sensores de una máquina de fresado en un proceso de producción. Tiene 10.000 filas y 14 columnas, sin nulos ni duplicados.

## Variables de entrada (features)

| Variable | Descripción |
|---|---|
| `Type` | Calidad del producto|
| `Air temperature [K]` | Temperatura del aire en kelvin|
| `Process temperature [K]` | Temperatura del proceso en kelvin |
| `Rotational speed [rpm]` | Velocidad de rotación de la herramienta, en revoluciones por minuto. |
| `Torque [Nm]` | Par de giro en newton-metro|
| `Tool wear [min]` | Minutos de desgaste acumulado de la herramienta. |

## Variable objetivo (target)

| Variable | Descripción |
|---|---|
| `Machine failure` | **1** si la máquina falla, **0** si no. Es lo que queremos predecir. |

## Variables identificadoras (se descartan)

| Variable | Descripción |
|---|---|
| `UDI` | Identificador único de cada fila (1 a 10.000). No aporta información predictiva. |
| `Product ID` | Código del producto: letra de calidad (L, M o H) más un número de serie. No aporta información predictiva. |

## Tipos de fallo

Indican qué tipo de fallo se ha producido, por lo que solo se conocen cuando el fallo ya ha ocurrido. Su relación con `Machine failure`, la variable objetivo, se analizará más adelante.

| Variable | Fallo |
|---|---|
| `TWF` | Desgaste de herramienta |
| `HDF` | Disipación de calor |
| `PWF` | Potencia |
| `OSF` | Sobreesfuerzo |
| `RNF` | Aleatorio |

### Conclusiones de la primera exploración

- **Tamaño:** el dataset tiene 10.000 registros y 14 columnas.
- **Valores nulos:** no hay valores nulos en ninguna columna (10.000 valores no nulos en todas y 0 % de nulos en el data dictionary).
- **Tipos de datos:** 9 columnas son enteras (`int64`), 3 son decimales (`float64`) y 2 son texto (`object`: `Product ID` y `Type`). Los tipos asignados son coherentes con el contenido de cada variable.
- **Variables identificadoras:** `UDI` y `Product ID` tienen 10.000 valores únicos, uno por registro, por lo que son identificadores y no aportan información para predecir.
- **Variable categórica:** `Type` tiene 3 categorías, por lo que habrá que codificarla antes de modelar.
- **Variables numéricas continuas:** `Air temperature [K]` y `Process temperature [K]` tienen pocos valores únicos (93 y 82), lo que indica que se miden con poca precisión decimal. `Rotational speed [rpm]`, `Torque [Nm]` y `Tool wear [min]` tienen más variedad (941, 577 y 246).
- **Variables binarias:** `Machine failure`, `TWF`, `HDF`, `PWF`, `OSF` y `RNF` tienen solo 2 valores únicos (0 y 1). `Machine failure` es la variable objetivo, y las otras cinco parecen indicar tipos de fallo, por lo que se estudiará su relación con el objetivo en la fase de análisis.